In [ ]:
# !pip install -U langchain langchain-google-genai langchain-milvus pymilvus sentence-transformers


In [1]:
# gemini_to_milvus_langchain.py
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
# Provider imports (package names may vary slightly across versions)
from langchain_google_genai import GoogleGenerativeAIEmbeddings
# Milvus vectorstore (choose the import matching your installed package)
# Option A (common): langchain_milvus
try:
    from langchain_milvus import Milvus as LangchainMilvus
except Exception:
    # Option B fallback: community import
    from langchain_community.vectorstores.milvus import Milvus as LangchainMilvus

import os


C:\Users\USER\Miniconda3\envs\kpn-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:

# ------------- CONFIG -------------
MILVUS_CONN = {"host": "127.0.0.1", "port": "19530"}   # local Milvus
COLLECTION_NAME = "milvus_langchain_test"
EMBEDDING_MODEL = "models/embedding-001"  # Gemini embedding model


In [3]:

# ------------- SAMPLE INDONESIAN TEXTS -------------
texts = [
    "Ini adalah ringkasan laporan keuangan Q1. Pendapatan naik 12% dibandingkan tahun lalu.",
    "Petunjuk instalasi: sambungkan kabel A ke port B dan pastikan LED menyala.",
    "Artikel: Perubahan iklim mempengaruhi produksi padi di Indonesia."
]


In [4]:
# ------------- 1) Chunk documents for better retrieval -------------
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
docs = []
for i, txt in enumerate(texts):
    chunks = splitter.split_text(txt)
    for j, c in enumerate(chunks):
        docs.append(Document(page_content=c, metadata={"source":"report.pdf", "page":1, "chunk": f"{i}-{j}"}))


In [6]:
# ------------- 2) Create Gemini embeddings wrapper (LangChain) -------------
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, google_api_key=os.environ["GEMINI_API_KEY"])  # reads GOOGLE_API_KEY env var


In [9]:
# ------------- 3) Create Milvus vectorstore from documents -------------
# This will create the collection in Milvus if it doesn't exist.
# connection_args is a dict; exact keys depend on langchain-milvus version (check docs).
vectorstore = LangchainMilvus.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    connection_args=MILVUS_CONN,
)

In [10]:
# ------------- 4) Retrieval example -------------
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
query = "Bagaimana performa pendapatan pada Q1?"
results = retriever.get_relevant_documents(query)

print("Top retrievals:")
for r in results:
    print(r.metadata, "->", r.page_content[:200])

C:\Users\USER\AppData\Local\Temp\ipykernel_5868\2419917717.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents(query)


Top retrievals:
{'chunk': '0-0', 'source': 'report.pdf', 'page': 1, 'pk': 460897216184195956} -> Ini adalah ringkasan laporan keuangan Q1. Pendapatan naik 12% dibandingkan tahun lalu.
{'chunk': '1-0', 'source': 'report.pdf', 'page': 1, 'pk': 460897216184195957} -> Petunjuk instalasi: sambungkan kabel A ke port B dan pastikan LED menyala.
{'chunk': '2-0', 'source': 'report.pdf', 'page': 1, 'pk': 460897216184195958} -> Artikel: Perubahan iklim mempengaruhi produksi padi di Indonesia.
